**OSTIn project: Develop software to automatically launch infrasound and satellite imagery analyses using the time and location provided by a VONA (Volcano Observatory Notice for Aviation).**




**Parse VONA messages to extract time and location data from both MAGMA (MAGMA VONA) and TVAAC (TVAAC VONA) in real time** 

Sources:
MAGMA VONA: https://magma.esdm.go.id/v1/vona 

TVAAC VONA: https://www.data.jma.go.jp/svd/vaac/data/vaac_list.html 

**Forecast Eruption Source Paramerters such as Mass Eruption Rate, Total Erupted Mass, Eruption Duration and percentage of fine ash particles based on VONA messages and Historical Volcanic Eruption Data using Bayesian statistics**


**VONA Monitoring Script**

This script continuously monitors the latest Volcanic Observatory Notice for Aviation (VONA) reports 
and detects changes in key volcanic activity parameters such as volcano name, ash cloud height above ground level (AGL),
and volcanic explosivity index (VEI) range. When changes are detected, the screen is cleared, and updated information is displayed.

Key Features:
- Fetches VONA reports from a specified data source (default: "TOKYO").
- Detects changes in volcano name, ash cloud height, and VEI range.
- Updates the display when changes occur.
- Uses the `ESP_INIT` function for eruption source parameter analysis when new data is available.
- Implements a progress bar to visualize the monitoring interval.

Dependencies:
- `vona_trigger`: Retrieves VONA reports.
- `re`: For text normalization.
- `esp_init.ESP_INIT`: Processes eruption source parameters.
- `time`: Introduces delays between monitoring cycles.
- `os`: Clears the terminal screen.
- `tqdm`: Provides a visual progress bar for monitoring intervals.

Contact: 
Dr. Mahmud Muhammad
Email: mahmud.muhammad@hotmail.com



In [ ]:
"""
VONA Monitoring Script

This script continuously monitors the latest Volcanic Observatory Notice for Aviation (VONA) reports 
and detects changes in key volcanic activity parameters such as volcano name, ash cloud height above ground level (AGL),
and volcanic explosivity index (VEI) range. When changes are detected, the screen is cleared, and updated information is displayed.

Key Features:
- Fetches VONA reports from a specified data source (default: "TOKYO").
- Detects changes in volcano name, ash cloud height, and VEI range.
- Updates the display when changes occur.
- Uses the `ESP_INIT` function for eruption source parameter analysis when new data is available.
- Implements a progress bar to visualize the monitoring interval.

Dependencies:
- `vona_trigger`: Retrieves VONA reports.
- `re`: For text normalization.
- `esp_init.ESP_INIT`: Processes eruption source parameters.
- `time`: Introduces delays between monitoring cycles.
- `os`: Clears the terminal screen.
- `tqdm`: Provides a visual progress bar for monitoring intervals.

Contact: 
Dr. Mahmud Muhammad
Email: mahmud.muhammad@hotmail.com

"""


import vona_trigger
import re
from esp_init import ESP_INIT
import time
import os
from tqdm import tqdm  # For progress bar

# Initialize storage for previous variable values
previous_values = {
    "volcano_name": None,
    "AshCloudHeight_AGL": None,
    "vei_range": None
}

def clear_screen():
    """Clears the terminal screen for a fresh output."""
    os.system('cls' if os.name == 'nt' else 'clear')

def normalize_text(text):
    """Removes extra spaces and normalizes text for meaningful comparison."""
    if isinstance(text, str):
        return re.sub(r'\s+', ' ', text).strip()  # Replace multiple spaces with one and trim
    return text  # Return non-string values as-is

while True:
    # Fetch the latest VONA data
    VONA_reports, VONA_latest_data, volcano_name, latitude, longitude, AshCloudHeight_AGL, vei_range = vona_trigger.VONA_Message(VONA_Source="MAGMA")
    
    
    
    # Store current values in a dictionary
    current_values = {
       
        "volcano_name": volcano_name,
        "AshCloudHeight_AGL": AshCloudHeight_AGL,
        "vei_range": vei_range
    }

   # Check for changes in each variable
    changes_detected = False
    for key in current_values:
        prev_value = normalize_text(previous_values[key]) if previous_values[key] is not None else None
        curr_value = normalize_text(current_values[key])
        
        if prev_value != curr_value:
            changes_detected = True
    
    # If changes are detected, clear screen and print new data
    
    if changes_detected:
        clear_screen()  # Clears previous prints and outputs
        print("🔔 **New VONA Data Detected:**\n")
        for key in current_values:
            prev_value = normalize_text(previous_values[key]) if previous_values[key] is not None else None
            curr_value = normalize_text(current_values[key])
            
            if prev_value != curr_value:
                print(f"🔹 **Change detected in {key}:**")
                print(f"   - **Previous:** {prev_value}")
                print(f"   - **New:** {curr_value}\n")
                previous_values[key] = current_values[key]  # Update stored value

        # Convert AshCloudHeight_AGL to km if it's not None
        if AshCloudHeight_AGL is not None or vei_range is  None:
            
            ESP_INIT(vei_levels=None, PlumeHeight=AshCloudHeight_AGL, show_plots=True)

        if vei_range is not None or AshCloudHeight_AGL is  None: 
            ESP_INIT(vei_levels=vei_range, PlumeHeight=None, show_plots=True)
        
        # if AshCloudHeight_AGL is not None and vei_range is not None:
        
        #     ESP_INIT(vei_levels=None, PlumeHeight=AshCloudHeight_AGL, show_plots=True)
        #     ESP_INIT(vei_levels=vei_range, PlumeHeight=None, show_plots=True)
        
        elif AshCloudHeight_AGL is  None and vei_range is  None:
            ESP_INIT(vei_levels=None, PlumeHeight=None, show_plots=True)
   
    # Progress bar with dynamic status
    status_message = "🟢 Waiting for next VONA report..." if changes_detected else "🟡 No new data - Monitoring..."

    with tqdm(total=1800, desc=status_message, unit="s", bar_format="{l_bar}{bar} {remaining}") as pbar:
        for _ in range(1800):
            time.sleep(1)  # Sleep in 1-second intervals
            pbar.update(1)  # Update progress bar



CSV file created: ./TVAAC_test\vaa_texts.csv
Scraping up to 1 pages for All Volcanoes...


🔄 Scraping Pages: 100%|██████████| 1/1 [00:04<00:00,  4.55s/page]


Scraping completed. Data saved at Magma_Test\vona_reports_sorted.csv
Extracting details from 15 reports...


📝 Extracting Reports:   0%|          | 0/15 [00:00<?, ?report/s]2025-05-09 11:23:02,597 - WARNING - Connection pool is full, discarding connection: magma.esdm.go.id. Connection pool size: 10
2025-05-09 11:23:02,601 - WARNING - Connection pool is full, discarding connection: magma.esdm.go.id. Connection pool size: 10
📝 Extracting Reports: 100%|██████████| 15/15 [00:02<00:00,  7.12report/s]



💾 Saving extracted reports...


...

                                   VEI and Related Data                                    
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ VEI ┃ Height_km.a.v.l_Percentile_90 ┃ Height_km.a.v.l_Percentile_95 ┃  Duration_hr_P5   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ 2.0 │      7.1995675465682805       │       7.406841181785159       │ 3.648028276657213 │
└─────┴───────────────────────────────┴───────────────────────────────┴───────────────────┘

                       VEI and Related Data                        
┏━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ VEI ┃  Duration_hr_P25  ┃  Duration_hr_P50  ┃  Duration_hr_P75  ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ 2.0 │ 2.980646704018757 │ 2.586750247676114 │ 2.251696440855864 │
└─────┴───────────────────┴───────────────────┴───────────────────┘

    VEI and Related Data    
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ VEI ┃  Duration_hr_P95   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ 2.0 │ 1.8393367642399199 │
└─────┴────────────────────┘

Model updated with xvar='MER_kg/s'
Model updated with yvar='MER_kg/s'.
Model updated with yvar='MASS_FRACTION_ASH_BELOW_63_micron'.
Saved plot to ESP_init_forVEI_[2]/Forecast_MASS_FRACTION_ASH_BELOW_63_micron\posterior_predictive_plot.png
Saved plot to ESP_init_forVEI_[2]/Forecast_MASS_FRACTION_ASH_BELOW_63_micron\percentiles_plot.png
Saved plot to ESP_init_forVEI_[2]/Forecast_MASS_FRACTION_ASH_BELOW_63_micron\contour_percentiles_plot.png
Saved DataFrame to ESP_init_forVEI_[2]/Forecast_MASS_FRACTION_ASH_BELOW_63_micron\forecast_MASS_FRACTION_ASH_BELOW_63_micron.csv
        MER_kg/s  MASS_FRACTION_ASH_BELOW_63_micron_Percentile_5  \
0  536979.456939                                        0.136359   

   MASS_FRACTION_ASH_BELOW_63_micron_Percentile_10  \
0                                         0.136541   

   MASS_FRACTION_ASH_BELOW_63_micron_Percentile_15  \
0                                         0.136666   

   MASS_FRACTION_ASH_BELOW_63_micron_Percentile_20  \
0                  

🟢 Waiting for next VONA report...:  17%|█▋         24:57